# **Import Dependencies**

In [ ]:
# data analysis and wrangling
import pandas as pd
import numpy as np
import random as rnd
# (Note: 'ydata_profiling' is commented out but left here for context)
# from ydata_profiling import ProfileReport

# visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# machine learning - general utilities
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler

# machine learning - models
from sklearn.linear_model import LogisticRegression, Perceptron, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

# **Loading Data**

In [ ]:
data = pd.read_csv('/kaggle/input/titanic/train.csv')
X_test = pd.read_csv('/kaggle/input/titanic/test.csv')

# **Feature Engineering**

## Cabin

In [ ]:
# ---------------------CABIN TRANSFORMER-------------------

class CabinTrans(BaseEstimator, TransformerMixin):
    def fit(self, df):
        return self

    def get_deck(self, cabin):
        cabin = str(cabin)
        if cabin=='nan':
            return 'N'
        else:
            return cabin[0]

    def group_deck(self, deck):
        if deck in ['A','B','T']:
            return 'ABT'  # Top 3 Decks
        elif deck in ['C','D','E']:
            return 'CDE'  # Next 3 Decks from top
        elif deck in ['F','G']:
            return 'FG'  # Bottom 2 Decks
        else:
            return 'N'

    def transform(self, df):
        df_final = df.copy()
        df_final['Cabin'] = df_final['Cabin'].apply(self.get_deck)
        df_final['Cabin'] = df_final['Cabin'].apply(self.group_deck)

        cabin_maps = {'ABT':0, 'CDE':1, 'FG':2, 'N':3}
        df_final['Cabin'] = df_final['Cabin'].map(cabin_maps)

        return df_final

## Title

In [ ]:
# ---------------------TITLE TRANSFORMER-------------------


class TitleTransformer(BaseEstimator, TransformerMixin):
    def fit(self, df):
        return self

    def title(self, name):
        return name.split(', ')[1].split('.')[0]

    def group(self, title):
        if title in ['Sir', 'Lady', 'the Countess', 'Jonkheer', 'Dona', 'Don', 'Dr', 'Major', 'Col', 'Capt', 'Rev']:
            return 'Special'

        elif title in ['Ms', 'Miss', 'Mlle']:
            return 'Miss'

        elif title in ['Mrs', 'Mme']:
            return 'Mrs'

        elif title in ['Mr', 'Master']:
            return 'Mr'

        else:
            return title


    def transform(self, df):
        df_final = df.copy()
        df_final['Title'] = df_final['Name'].apply(self.title)
        df_final['Title'] = df_final['Title'].apply(self.group)

        title_mapping = {"Special": 1, "Miss": 2, "Mrs": 3, "Mr": 4}
        df_final['Title'] = df_final['Title'].map(title_mapping)
        df_final['Title'] = df_final['Title'].fillna(0)
        df_final = df_final.drop('Name', axis=1)

        return df_final

## Age

In [ ]:
# ---------------------AGE TRANSFORMER-------------------

class AgeTransformer(BaseEstimator, TransformerMixin):
    def fit(self, df):
        return self

    def title(self, name):
        return name.split(', ')[1].split('.')[0]

    def group(self, title):
        if title in ['Sir', 'Lady', 'the Countess', 'Jonkheer', 'Dona', 'Don', 'Dr', 'Major', 'Col', 'Capt', 'Rev']:
            return 'Special'

        else:
            return title

    def transform(self, df):
        df_final = df.copy()
        # Extract title before 'Name' is potentially dropped by another transformer
        df_final['Title-For-Age'] = df_final['Name'].apply(self.title)
        df_final['Title-For-Age'] = df_final['Title-For-Age'].apply(self.group)

        df_final['Age'] = df_final.groupby('Title-For-Age')['Age'].transform(lambda x: x.fillna(x.median()))
        df_final = df_final.drop('Title-For-Age', axis=1)

        return df_final

## Family

In [ ]:
# ---------------------FAMILY TRANSFORMER-------------------

class FamilyTransformer(BaseEstimator, TransformerMixin):
    def fit(self, df):
        return self

    def transform(self, df):
        df_final = df.copy()
        df_final['FamilySize'] = df_final['SibSp'] + df_final['Parch'] + 1
        df_final = df_final.drop(['SibSp','Parch'], axis=1)

        df_final['IsAlone'] = (df_final['FamilySize'] == 1).astype(int)

        df_final['Age*Class'] = df_final.Age * df_final.Pclass

        return df_final

## Feature Creation

In [ ]:
# ---------------------FARE FEATURE TRANSFORMER-------------------

class FeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, df, y=None):
        return self

    def transform(self, df):
        df_final = df.copy()

        # Ensure FamilySize exists
        if 'FamilySize' not in df_final.columns:
            df_final['FamilySize'] = df_final['SibSp'] + df_final['Parch'] + 1

        # Handle missing values safely
        df_final['Fare'] = df_final.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
        df_final['Pclass'] = df_final['Pclass'].fillna(3)
        df_final['FamilySize'] = df_final['FamilySize'].replace(0, 1)

        # Create simple interaction features
        df_final['FarePerClass'] = df_final['Fare'] / df_final['Pclass']
        df_final['FarePerClassBand'] = pd.cut(
            df_final['FarePerClass'],
            bins=[-0.001, 170.7764, 341.5528, 512.3292],
            labels=[0, 1, 2]
        ).astype(float)


        df_final['FamilySizePclass'] = df_final['FamilySize'] * df_final['Pclass']
        df_final['FamilySizePclassBand'] = pd.cut(
            df_final['FamilySizePclass'],
            bins=[-0.001, 11.66666667, 22.33333333, 33.],
            labels=[0, 1, 2]
        ).astype(float)

        return df_final

## Fare

In [ ]:
# ---------------------FARE TRANSFORMER-------------------

class FareTransformer(BaseEstimator, TransformerMixin):
    def fit(self, df):
        return self

    def transform(self, df):
        df_final = df.copy()

        df_final['FareBand'] = pd.cut(
            df_final['Fare'],
            bins=[-0.001, 128.0823, 256.1646, 384.2469, 512.3292],
            labels=[0, 1, 2, 4]
        ).astype(float)

        return df_final

## Ticket

In [ ]:
# ---------------------TICKET TRANSFORMER-------------------

class TicketTransformer(BaseEstimator, TransformerMixin):
    def fit(self, df):
        return self

    def get_prefix(self, ticket):
        parts = str(ticket).split()
        if len(parts) > 1 and not parts[0].isdigit():
            return parts[0]
        else:
            return 'NONE'

    def simplify_prefix(self, prefix):
        prefix = prefix.upper()
        prefix = prefix.replace('.', '').replace('/', '')
        if prefix in ['STONO', 'SOTONOQ', 'SOTONO2', 'A5', 'A4']:
            return 'Low'
        elif prefix in ['PC', 'CA', 'SC', 'SCPARIS', 'F', 'FCC']:
            return 'High'
        elif prefix in ['WEP', 'SOP', 'SOC', 'SPP', 'SWPP']:
            return 'Crew'
        else:
            return 'None'

    def transform(self, df):
        df_final = df.copy()
        df_final['TicketPrefix'] = df_final['Ticket'].apply(self.get_prefix)
        df_final['TicketPrefix'] = df_final['TicketPrefix'].apply(self.simplify_prefix)

        ticket_mapping = {"Low": 0, "High": 1, "Crew": 2, "None": 3}
        df_final['TicketPrefix'] = df_final['TicketPrefix'].map(ticket_mapping)
        df_final['TicketPrefix'] = df_final['TicketPrefix'].fillna(0)
        df_final = df_final.drop('Ticket', axis=1)

        return df_final

## Categorical Encoding

In [ ]:
# ---------------------CATEGORICAL TRANSFORMER-------------------

class CategoricalTransformer(BaseEstimator, TransformerMixin):
    def fit(self, df):
        return self

    def transform(self, df):
        df_final = df.copy()
        # Impute missing values
        df_final[['Sex','Embarked']] = SimpleImputer(strategy='most_frequent').fit_transform(df_final[['Sex','Embarked']])
        # Map Sex to numerical values
        df_final['Sex'] = df_final['Sex'].map({'male':0, 'female':1})
        # Reshape Embarked and apply OrdinalEncoder
        df_final['Embarked'] = OrdinalEncoder().fit_transform(df_final[['Embarked']]) # Reshape to 2D
        return df_final

# **Pipeline**

In [ ]:
y = data['Survived']
X = data.drop('Survived', axis=1)

In [ ]:
preprocess_pipeline = Pipeline([
    ('cabin', CabinTrans()),
    ('age', AgeTransformer()),
    ('title', TitleTransformer()),
    ('family', FamilyTransformer()),
    ('fare_features', FeatureTransformer()),
    ('fare', FareTransformer()),
    ('ticket', TicketTransformer()),
    ('encoding', CategoricalTransformer()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=8))
])

In [ ]:
data_processed = preprocess_pipeline.fit_transform(X)
print('shape of processed data:', data_processed.shape)

In [ ]:
data.head()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(data_processed, y, test_size=0.2, random_state=42)

# **Model**

In [ ]:
models = {'LogReg': LogisticRegression(),
         'XGBoost': XGBClassifier(),
         'DecisionTree': DecisionTreeClassifier(),
         'RandomForest': RandomForestClassifier(n_estimators=100),
          'AdaBoost': AdaBoostClassifier(),
          'GradBoost': GradientBoostingClassifier(),
         'KNN': KNeighborsClassifier(n_neighbors=3),
         'LinearSVC': LinearSVC(max_iter=1500)}

res1, res2 = [], []
for i in models.keys():
    model = models[i]
    model.fit(X_train, y_train)
    res1.append(model.score(X_train, y_train))
    res2.append(cross_val_score(model, X_train, y_train).mean())
pd.DataFrame({'Models':models.keys(), 'Training Score':res1, 'Cross val Score':res2})

## Random Forest Classifier

### V1

In [ ]:
param_grid = [{
    'n_estimators': [100, 200, 300],
    'max_features': [None, "sqrt", "log2"],
    'max_depth': [3,5,7,10,12],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [5],
}]

forest = RandomForestClassifier()
grid_search = GridSearchCV(forest, param_grid, cv=5, verbose=0, n_jobs=-1)
grid_search.fit(X_train, y_train)

In [ ]:
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
forest = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=20,
    min_samples_leaf=5,
    max_features='sqrt',
    max_depth=5,
    bootstrap=True
)

cv = cross_val_score(forest, X_train, y_train, cv=5)
print(cv, '\nmean:', cv.mean())

### V2

In [ ]:
param_grid_v2 = [{
    'n_estimators': [200, 300, 400],
    'max_depth': [5, 8, 12, 16, None],
    'max_features': [None, 'sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True],
}]

forest_v2 = RandomForestClassifier(random_state=42, n_jobs=-1)

# grid_search_v2 = GridSearchCV(
#     forest_v2, 
#     param_grid_v2, 
#     cv=5, 
#     scoring='accuracy',
#     verbose=2,
#     n_jobs=-1
# )


# grid_search_v2.fit(X_train, y_train)

In [ ]:
# print(grid_search_v2.best_params_)
# print(grid_search_v2.best_score_)

In [ ]:
forest = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features='log2',
    max_depth=5,
    bootstrap=True
)

cv = cross_val_score(forest, X_train, y_train, cv=5)
print(cv, '\nmean:', cv.mean())

### V3

In [ ]:
best_rf = RandomForestClassifier(
    bootstrap=True,
    max_depth=5,
    max_features='sqrt',
    min_samples_leaf=2,
    min_samples_split=8,
    n_estimators=220,
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(best_rf, X_train, y_train, cv=5, scoring='accuracy')
print(cv_scores)
print('Mean CV Score:', cv_scores.mean())

### V4

In [ ]:
best_rf_v4 = RandomForestClassifier(
    bootstrap=True,
    max_depth=15,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=3,
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
cv_scores = cross_val_score(best_rf, X_train, y_train, cv=5, scoring='accuracy')
print(cv_scores)
print('Mean CV Score:', cv_scores.mean())

## XGBoost

### V1

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 10],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1],
    'gamma': [0, 1],
    'min_child_weight': [1, 3]
}


In [ ]:
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# grid_search_xgb = GridSearchCV(
#     estimator=xgb,
#     param_grid=param_grid_xgb,
#     scoring='accuracy',
#     cv=cv,
#     n_jobs=-1,
#     verbose=1
# )

# grid_search_xgb.fit(
#     X_train,
#     y_train,
#     eval_set=[(X_val, y_val)],
#     early_stopping_rounds=50,
#     verbose=False
# )

In [ ]:
# print(grid_search_xgb.best_params_)
# print(grid_search_xgb.best_score_)

### V2

In [ ]:
param_grid_refined = {
    'n_estimators': [100, 150, 200],
    'learning_rate': [0.1, 0.15, 0.2],
    'max_depth': [3, 4, 5],
    'min_child_weight': [1, 2],
    'gamma': [0, 0.1, 0.2],
    'subsample': [0.9, 1],
    'colsample_bytree': [0.8, 0.9, 1],
    'reg_lambda': [0.8, 1, 1.2],
    'reg_alpha': [0, 0.05, 0.1]
}

In [ ]:
# random_search_xgb = RandomizedSearchCV(
#     estimator=xgb,
#     param_distributions=param_grid_refined,
#     n_iter=40,
#     scoring='accuracy',
#     cv=cv,
#     n_jobs=-1,
#     verbose=1,
#     random_state=42
# )
# random_search_xgb.fit(
#     X_train,
#     y_train,
#     eval_set=[(X_val, y_val)],
#     early_stopping_rounds=50,
#     verbose=False
# )

In [ ]:
# print(random_search_xgb.best_params_)
# print(random_search_xgb.best_score_)

### Final Version

In [ ]:
best_xgb = XGBClassifier(
    subsample=1,
    reg_lambda=1.2,
    reg_alpha=0.05,
    n_estimators=102,
    min_child_weight=2,
    max_depth=3,
    learning_rate=0.2,
    gamma=0.1,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
cv_scores = cross_val_score(best_xgb, X_train, y_train, cv=5)
print(cv_scores)
print('Mean CV Score:', cv_scores.mean())

## LinearSVC

In [ ]:
# pipe = make_pipeline(
#     StandardScaler(), 
#     LinearSVC(random_state=42, max_iter=20000)
# )

In [ ]:
# param_grid_svc = {
#     'linearsvc__C': [0.01, 0.1, 0.5, 1, 2, 5, 10],
#     'linearsvc__loss': ['hinge', 'squared_hinge']
# }

# grid_svc = GridSearchCV(
#     pipe,
#     param_grid_svc,
#     cv=5,
#     scoring='accuracy',
#     verbose=2,
#     n_jobs=-1
# )

# grid_svc.fit(X_train, y_train)

In [ ]:
# print(grid_svc.best_params_)
# print(grid_svc.best_score_)

# Ensemble

In [ ]:
ensemble = VotingClassifier(
    estimators=[('rf', best_rf_v4), ('xgb', best_xgb)],
    voting='soft',
    weights=[0.7, 0.3]
)

cv_scores = cross_val_score(ensemble, X_train, y_train, cv=5, scoring='accuracy')
print(cv_scores)
print('Mean CV Score:', cv_scores.mean())

In [ ]:
stack = StackingClassifier(
    estimators=[
        ('rf', best_rf_v4),
        ('xgb', best_xgb)
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

stack.fit(X_train, y_train)
cv_scores = cross_val_score(stack, X_train, y_train, cv=5, scoring='accuracy')
print('Mean CV Score:', cv_scores.mean())

# **Submission**

In [ ]:
model = ensemble


model.fit(data_processed, y)

In [ ]:
test_processed = preprocess_pipeline.transform(X_test)

In [ ]:
pred = model.predict(test_processed)

In [ ]:
submission = pd.DataFrame({'PassengerId':X_test['PassengerId'], 'Survived':pred})
submission.to_csv('Submission.csv', index=False)

In [ ]:
pd.read_csv('Submission.csv')